### RAG pipleline - Data Ingestion to Vector DB PileLine

In [35]:
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader,DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [36]:
def load_pdf_documents(pdf_directory: str) -> list[Document]:
    all_docs = []
    pdf_path = Path(pdf_directory)

    pdf_files = list(pdf_path.glob("**/*.pdf"))
    print(f"I found {len(pdf_files)}")
    for pdf_file in pdf_files:
        print(f"\n Processing :{pdf_file.name}")
        try:
            loader = PyMuPDFLoader(pdf_file)
            documents = loader.load()

            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"

            all_docs.extend(documents)
            print(f"loaded {len(documents)} pages")

        except Exception as e:
            print(f"error : {e}")
    return all_docs

In [37]:
all_pdf_docs = load_pdf_documents("data/pdf_files")


I found 3

 Processing :AI engineering Interview.pdf
loaded 20 pages

 Processing :HR interview.pdf
loaded 12 pages

 Processing :Type script interview.pdf
loaded 20 pages


### Text splitting get to chunks


In [38]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, 
        chunk_overlap=chunk_overlap,
        length_function = len,
        separators = ["\n\n","\n"," ",""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    if split_docs:
        print(f"\n Example chunk:")
        print(f"Content : {split_docs[0].page_content[:200]} ...")
        print(f"Metadata : {split_docs[0].metadata} ...")
    return split_docs

In [40]:
chunks = split_documents(all_pdf_docs, chunk_size=1000, chunk_overlap=200)

Split 52 documents into 126 chunks

 Example chunk:
Content : Absolutely. Below is a 100-question AI Engineering interview set based directly on the skills you provided. I also added 
dedicated questions about LangGraph and smolagents, especially around agents,  ...
Metadata : {'producer': 'Skia/PDF m156 Google Docs Renderer', 'creator': '', 'creationdate': '', 'source': 'data\\pdf_files\\AI engineering Interview.pdf', 'file_path': 'data\\pdf_files\\AI engineering Interview.pdf', 'total_pages': 20, 'format': 'PDF 1.4', 'title': 'AI engineering Interview', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0, 'source_file': 'AI engineering Interview.pdf', 'file_type': 'pdf'} ...
